# Experiment 16 - Model Blending

This experiment combines predictions from XGBoost, LightGBM, and CatBoost to test whether different boosting models make complementary predictions.

Current local best: **0.941776**

Current Kaggle benchmark: **0.941680**

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'train.csv'

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=['Will_Buy_EV', 'id'])
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print('Data and preprocessing ready.')
print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))

Data and preprocessing ready.
Training rows: 534932
Validation rows: 133733


In [2]:
xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

cat_model = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=False,
    thread_count=-1
)

print('Models created.')

Models created.


In [3]:
xgb_model.fit(X_train_processed, y_train)
lgb_model.fit(X_train_processed, y_train)
cat_model.fit(X_train_processed, y_train)

xgb_pred = xgb_model.predict_proba(X_valid_processed)[:, 1]
lgb_pred = lgb_model.predict_proba(X_valid_processed)[:, 1]
cat_pred = cat_model.predict_proba(X_valid_processed)[:, 1]

xgb_auc = roc_auc_score(y_valid, xgb_pred)
lgb_auc = roc_auc_score(y_valid, lgb_pred)
cat_auc = roc_auc_score(y_valid, cat_pred)

print('=' * 60)
print('INDIVIDUAL MODEL RESULTS')
print('=' * 60)
print(f'XGBoost:  {xgb_auc:.6f}')
print(f'LightGBM: {lgb_auc:.6f}')
print(f'CatBoost: {cat_auc:.6f}')

INDIVIDUAL MODEL RESULTS
XGBoost:  0.941776
LightGBM: 0.941493
CatBoost: 0.941389


In [4]:
blends = {
    'XGB_80_Cat_20': 0.80 * xgb_pred + 0.20 * cat_pred,
    'XGB_70_Cat_30': 0.70 * xgb_pred + 0.30 * cat_pred,
    'XGB_60_Cat_40': 0.60 * xgb_pred + 0.40 * cat_pred,
    'XGB_80_LGB_20': 0.80 * xgb_pred + 0.20 * lgb_pred,
    'XGB_70_LGB_30': 0.70 * xgb_pred + 0.30 * lgb_pred,
    'XGB_60_LGB_40': 0.60 * xgb_pred + 0.40 * lgb_pred,
    'XGB_60_Cat_20_LGB_20': 0.60 * xgb_pred + 0.20 * cat_pred + 0.20 * lgb_pred,
    'XGB_70_Cat_15_LGB_15': 0.70 * xgb_pred + 0.15 * cat_pred + 0.15 * lgb_pred
}

results = []

for name, predictions in blends.items():
    score = roc_auc_score(y_valid, predictions)
    results.append({'Blend': name, 'ROC-AUC': score})

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

previous_best = 0.941776
kaggle_benchmark = 0.941680
best_blend_score = results_df.iloc[0]['ROC-AUC']

print('=' * 60)
print('EXPERIMENT 16 RESULTS')
print('=' * 60)
print(results_df.to_string(index=False))
print()
print(f'Previous local best: {previous_best:.6f}')
print(f'Best blend: {best_blend_score:.6f}')
print(f'Difference vs local best: {best_blend_score - previous_best:+.6f}')
print(f'Kaggle benchmark: {kaggle_benchmark:.6f}')
print(f'Difference vs Kaggle benchmark: {best_blend_score - kaggle_benchmark:+.6f}')

if best_blend_score > previous_best:
    print('\nNEW LOCAL BEST MODEL')
else:
    print('\nNo blend beat the current local best.')

EXPERIMENT 16 RESULTS
               Blend  ROC-AUC
       XGB_70_LGB_30 0.941815
XGB_70_Cat_15_LGB_15 0.941813
       XGB_80_LGB_20 0.941813
XGB_60_Cat_20_LGB_20 0.941809
       XGB_60_LGB_40 0.941805
       XGB_80_Cat_20 0.941791
       XGB_70_Cat_30 0.941781
       XGB_60_Cat_40 0.941760

Previous local best: 0.941776
Best blend: 0.941815
Difference vs local best: +0.000039
Kaggle benchmark: 0.941680
Difference vs Kaggle benchmark: +0.000135

NEW LOCAL BEST MODEL
